In [5]:
import os
from dotenv import load_dotenv

#load_dotenv(dotenv_path='.env')
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError(".env 파일에 GROQ_API_KEY 가 없습니다.")
print("GROQ_API_KEY 설정 확인 완료", GROQ_API_KEY[:2])


GROQ_API_KEY 설정 확인 완료 gs


#### Groq API로 호출 가능한 모델 목록 확인

- Groq는 OpenAI 호환 API이므로 `openai` SDK에 `base_url`만 Groq 주소로 지정하면 `client.models.list()`로 모델 목록을 조회할 수 있습니다.
- 목록 조회는 토큰을 사용하지 않습니다.
- 아래 `ChatOpenAI`에서 사용할 모델명이 목록에 있는지 확인하는 용도로도 사용할 수 있습니다.

In [6]:
# openai SDK 불러오기 (Groq도 OpenAI 호환 API라서 같은 SDK 사용)
from openai import OpenAI

GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트

# Groq 키와 엔드포인트로 클라이언트 생성 (GROQ_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)

# 호출 가능한 모델 목록 조회 (id 기준 오름차순 정렬)
models = sorted(client.models.list().data, key=lambda m: m.id)
print(f"조회된 모델 수: {len(models)}\n")
for m in models:
    print(f"{m.id:<45} {m.owned_by}")

조회된 모델 수: 11

allam-2-7b                                    SDAIA
canopylabs/orpheus-arabic-saudi               Canopy Labs
canopylabs/orpheus-v1-english                 Canopy Labs
meta-llama/llama-prompt-guard-2-22m           Meta
meta-llama/llama-prompt-guard-2-86m           Meta
openai/gpt-oss-120b                           OpenAI
openai/gpt-oss-20b                            OpenAI
openai/gpt-oss-safeguard-20b                  OpenAI
qwen/qwen3.8-27b                              Alibaba Cloud
whisper-large-v3                              OpenAI
whisper-large-v3-turbo                        OpenAI


In [7]:
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_messages(
    [ ("system", "당신은 개발자입니다.") , 
     ("user", "{input}") ]
)
print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [8]:
prompt_text = prompt.format(input="파이썬은 무엇인가요? 자세하게 설명해주세요")
print(type(prompt_text))
print(prompt_text)

<class 'str'>
System: 당신은 개발자입니다.
Human: 파이썬은 무엇인가요? 자세하게 설명해주세요


In [9]:
from langchain_openai import ChatOpenAI 

#llm = ChatOpenAI(api_key=OPENAI_API_KEY, model_name="gpt-3.5-turbo-0125")

# Groq API를 사용하는 ChatOpenAI 인스턴스 생성
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",
    # model="qwen/qwen3.8-27b",
    temperature=0.7
)
print(type(llm))
print(llm.model_name)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
openai/gpt-oss-120b


In [ ]:
try:
    response = llm.invoke(prompt_text)
    print(type(response))
    print("응답:", response.content)
except Exception as e:
    print(f"오류 발생: {e}")

## LCEL(LangChain Expression Language)
### Prompt + LLM 모델 + OutputParser

In [11]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [12]:
chain = prompt | llm | output_parser
print(type(chain))
print(chain)

<class 'langchain_core.runnables.base.RunnableSequence'>
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]) middle=[ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002B24BCC1370>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002B24C463AD0>, root_client=<openai.OpenAI object at 0x000002B24C425310>, root_async_client=<openai.AsyncOpenAI object at 0x000002B24C463650>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, ope

In [13]:
response = chain.invoke({"input":"LangChain은 무엇인가요?"})

In [14]:
print(type(response)) #Str
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
**LangChain**은 **대규모 언어 모델(LLM)**을 활용한 **애플리케이션**을 보다 **쉽고 체계적으로** 구축할 수 있도록 도와주는 **오픈소스 프레임워크**입니다.  

---

## 1. 핵심 목표
- **LLM을 단독으로 쓰는 수준을 넘어**: 프롬프트 관리, 체인(Chain) 구성, 메모리, 도구 연동 등 복합적인 로직을 손쉽게 구현한다.
- **다양한 백엔드와 연동**: OpenAI, Anthropic, Cohere, HuggingFace 등 여러 LLM 제공자를 지원한다.
- **모듈화·재사용성**: 기본 제공되는 컴포넌트를 조합해 복잡한 워크플로우를 빠르게 만들 수 있다.

---

## 2. 주요 구성 요소

| 컴포넌트 | 역할 | 예시 |
|--------|------|------|
| **LLM** | 실제 텍스트를 생성·예측하는 모델 | `OpenAI`, `ChatOpenAI`, `Anthropic` 등 |
| **PromptTemplate** | 프롬프트를 템플릿화·파라미터화 | `"Answer the question: {question}"` |
| **Chain** | 여러 단계(프롬프트 → LLM → 후처리)를 순차적으로 연결 | `LLMChain`, `SequentialChain`, `RouterChain` |
| **Memory** | 대화·세션 상태를 기억하고 재활용 | `ConversationBufferMemory`, `VectorStoreRetrieverMemory` |
| **Retriever / VectorStore** | 문서 검색·유사도 기반 정보 회수 | `FAISS`, `Pinecone`, `ElasticSearch` |
| **Tool** | LLM이 외부 API·함수를 호출하도록 연결 | `SearchTool`, `SQLTool`, 사용자 정의 함수 |
| **Agent** | “프롬프트 → 도구 선택 →